In [1]:
import pandas as pd
import numpy as np

# analysis window
START_DATE = pd.Timestamp("2023-12-29")
END_DATE   = pd.Timestamp("2025-12-31")
sec_px_csv = "sec_px.csv" # path to security price data
sec_metadata_csv = "sec_metadata.csv" # path to security metadata


## Part A - Top 3 & Bottom 3 securities by annualised TWR 

In [2]:
def load_sec_px(path: str) -> pd.DataFrame:
    px = pd.read_csv(
        path,
        parse_dates=["date"],
        dayfirst=False
    )
    px = px.set_index("date").sort_index()
    px = px.apply(pd.to_numeric, errors="coerce")
    return px
sec_px = load_sec_px(sec_px_csv)

def restrict_window(px: pd.DataFrame,
                    start: pd.Timestamp,
                    end: pd.Timestamp) -> pd.DataFrame:
    return px.loc[(px.index >= start) & (px.index <= end)]
sec_px_window = restrict_window(sec_px, START_DATE, END_DATE)

def filter_eligible_securities(px: pd.DataFrame) -> pd.DataFrame:
    start_prices = px.iloc[0]
    end_prices   = px.iloc[-1]

    eligible = start_prices.notna() & end_prices.notna()
    px = px.loc[:, eligible]

    returns = px.pct_change()
    has_returns = returns.notna().sum() > 0

    return px.loc[:, has_returns]
sec_px_clean = filter_eligible_securities(sec_px_window)

In [3]:
def load_sec_metadata(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path)
    meta.columns = meta.columns.str.lower()
    meta["date"] = pd.to_datetime(meta["date"])
    return meta
sec_metadata = load_sec_metadata(sec_metadata_csv)
def build_name_map(meta: pd.DataFrame) -> pd.Series:
    latest = (
        meta.sort_values("date")
            .groupby("sedol")
            .tail(1)
    )
    return latest.set_index("sedol")["name"]

name_map = build_name_map(sec_metadata)
name_map.head()

sedol
TS150329                               Far East Horizon (Det)
BD5CK8      CETC Cyberspace Security Technology Co. Ltd. C...
BD5CK3      Yuan Longping High-Tech Agriculture Co., Ltd. ...
BD5CKR                            SeeWay.ai Co., Ltd. Class A
BD5LY1                 Apeloa Pharmaceutical Co. Ltd. Class A
Name: name, dtype: object

In [4]:
def compute_annualised_twr(px: pd.DataFrame) -> pd.DataFrame:
    """
    Compute annualised Time-Weighted Return (TWR) for each security
    using monthly price data.

    Parameters
    ----------
    px : pd.DataFrame
        Cleaned price panel with DatetimeIndex and SEDOL columns

    Returns
    -------
    pd.DataFrame
        Columns: sedol, annualised_twr, n_months
    """
    # Monthly returns
    returns = px.pct_change()

    results = []

    for sedol in returns.columns:
        r = returns[sedol].dropna()

        if len(r) == 0:
            continue

        # Total time-weighted return
        twr_total = (1 + r).prod() - 1

        # Annualise based on number of months observed
        annualised_twr = (1 + twr_total) ** (12 / len(r)) - 1

        results.append({
            "sedol": sedol,
            "annualised_twr": annualised_twr,
            "n_months": len(r)
        })

    return pd.DataFrame(results)

twr_df = compute_annualised_twr(sec_px_clean)


In [5]:
twr_df["annualised_twr"].describe()
# twr_df["n_months"].describe()

# twr_df["n_months"].min(), twr_df["n_months"].max()

count    814.000000
mean       0.180755
std        0.332745
min       -0.436901
25%       -0.019264
50%        0.118526
75%        0.312294
max        2.946943
Name: annualised_twr, dtype: float64

In [6]:
def attach_security_names(twr_df: pd.DataFrame,
                          name_map: pd.Series) -> pd.DataFrame:
    out = twr_df.copy()
    out["name"] = out["sedol"].map(name_map)
    return out
twr_df_named = attach_security_names(twr_df, name_map)

In [7]:
# Get final result
def get_top_bottom_securities(twr_named: pd.DataFrame,
                              n: int = 3) -> tuple[pd.DataFrame, pd.DataFrame]:
    ranked = twr_named.dropna(subset=["name"])

    top = (
        ranked.sort_values("annualised_twr", ascending=False)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    bottom = (
        ranked.sort_values("annualised_twr", ascending=True)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    return top, bottom
top_securities, bottom_securities = get_top_bottom_securities(twr_df_named)
print("Top 3 Securities:")
print(top_securities)   
print("\nBottom 3 Securities:")
print(bottom_securities)

Top 3 Securities:
                                                  name  annualised_twr
325  Victory Giant Technology (HuiZhou) Co., Ltd. C...        2.946943
326            Eoptolink Technology Inc., Ltd. Class A        2.488145
504          Cambricon Technologies Corp. Ltd. Class A        2.152435

Bottom 3 Securities:
                                                  name  annualised_twr
198  Chongqing Zhifei Biological Products Co., Ltd....       -0.436901
484               Hygeia Healthcare Holdings Co., Ltd.       -0.413385
7                   Legend Biotech Corp. Sponsored ADR       -0.406499


## PART B - Cumulative Returns for each sector

In [ ]:
def compute_sector_cumulative_return(sec_px, sec_metadata):

    # ── Step 1: Reshape sec_px wide → long ──────────────────────────────
    px = sec_px.reset_index().rename(columns={'index': 'date'})
    px_long = px.melt(id_vars='date', var_name='sedol', value_name='price')
    px_long = px_long.sort_values(['sedol', 'date'])

    # ── Step 2: Filter date range ────────────────────────────────────────
    px_long = px_long[
        (px_long['date'] >= '2023-12-01') &
        (px_long['date'] <= '2025-12-31')
    ]

    # ── Step 3: Compute monthly returns per stock ────────────────────────
    px_long['return'] = px_long.groupby('sedol')['price'].pct_change()
    px_long = px_long.dropna(subset=['return'])

    # ── Step 4: Convert both dates to year-month period ──────────────────
    px_long['ym'] = pd.to_datetime(px_long['date']).dt.to_period('M')

    meta = sec_metadata[['sedol', 'sector', 'weight', 'date']].copy()
    meta['ym'] = pd.to_datetime(meta['date']).dt.to_period('M')

    # ── Step 5: Lag metadata by 1 month (shift period, not date) ─────────
    meta['ym'] = meta['ym'] + 1
    # Dec 2023 weight → ym becomes Jan 2024 → merges with Jan 2024 return ✅

    # ── Step 6: Merge on sedol + year-month ──────────────────────────────
    df = px_long.merge(meta[['sedol', 'ym', 'sector', 'weight']], 
                       on=['sedol', 'ym'], how='inner')
    print("Merged shape:", df.shape)  # should be ~19,000+

    # ── Step 7: Normalize weights within each sector-month ───────────────
    df['w_norm'] = df.groupby(['sector', 'ym'])['weight'].transform(
        lambda x: x / x.sum()
    )

    # ── Step 8: Compute weighted return per sector per month ─────────────
    df['weighted_return'] = df['w_norm'] * df['return']
    monthly_sector = (
        df.groupby(['sector', 'ym'])['weighted_return']
        .sum()
        .reset_index()
    )

    # ── Step 9: Chain-link monthly returns per sector ────────────────────
    monthly_sector['gross_return'] = 1 + monthly_sector['weighted_return']
    sector_cum = (
        monthly_sector.groupby('sector')['gross_return']
        .prod()
        .reset_index()
    )
    sector_cum['cumulative_return'] = sector_cum['gross_return'] - 1
    
    # DEBUGGING
    # How many months does it appear in each dataset?
    print("In px_long:", px_long[px_long['sedol'] == 'BMC6XV']['ym'].tolist())
    print("In meta:", meta[meta['sedol'] == 'BMC6XV']['ym'].tolist())
    # Check how many stocks and what the monthly returns look like
    print("Materials stock count:", df[df['sector'] == 'Materials']['sedol'].nunique())
    print(monthly_sector[monthly_sector['sector'] == 'Materials'].sort_values('weighted_return'))


    return sector_cum[['sector', 'cumulative_return']].sort_values(
        'cumulative_return', ascending=False
    )
sector_cumulative_returns = compute_sector_cumulative_return(sec_px_clean, sec_metadata)
print("\nSector Cumulative Returns:")
print(sector_cumulative_returns)

Merged shape: (14807, 7)
In px_long: [Period('2024-01', 'M'), Period('2024-02', 'M'), Period('2024-03', 'M'), Period('2024-04', 'M'), Period('2024-05', 'M'), Period('2024-06', 'M'), Period('2024-07', 'M'), Period('2024-08', 'M'), Period('2024-09', 'M'), Period('2024-10', 'M'), Period('2024-11', 'M'), Period('2024-12', 'M'), Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-03', 'M'), Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M'), Period('2025-12', 'M')]
In meta: [Period('2025-12', 'M'), Period('2026-01', 'M')]
Materials stock count: 110
        sector       ym  weighted_return  gross_return
192  Materials  2024-01        -0.101593      0.898407
207  Materials  2025-04        -0.062806      0.937194
198  Materials  2024-07        -0.062382      0.937618
201  Materials  2024-10        -0.046907      0.953093
199  Materials  2024-08   

In [ ]:
    # Monthly returns should mostly be between -30% and +30%
    # print(df['return'].describe())
    # print("Returns > 90% in one month:", (df[df['return'] > 0.9]))
    # print("Returns < -50% in one month:", df[df['return'] < -0.5])

    # # Check 1 — are returns per stock reasonable?
    # stock_total_returns = (
    #     df.groupby('sedol')['return']
    #     .apply(lambda x: (1 + x).prod() - 1)
    #     .sort_values(ascending=False)
    # )
    # print("Top 10 stock cumulative returns:")
    # print(stock_total_returns.head(10))
    # print("\nBottom 10 stock cumulative returns:")
    # print(stock_total_returns.tail(10))

    # # Check 2 — what does the monthly sector return look like?
    # monthly_sector[monthly_sector['sector'] == 'Materials'] \
    #     .sort_values('weighted_return', ascending=False)

    # Check 3 — check a known stock's price series
    # e.g., BYD sedol is 653665
    # print(df[df['sedol'] == 'BMC6XV'][['date', 'price', 'return']])

    # print("px_long shape:", px_long.shape)
    # print("meta shape:", meta.shape)
    # print("merged df shape:", df.shape)
    # print("date sample from px_long:", px_long['date'].unique())
    # print("date sample from meta:", meta['date'].unique())

    # # Check which dates exist in each
    # print("px_long dates:", sorted(px_long['date'].unique()))
    # print("meta dates (after lag):", sorted(meta['date'].unique()))

    # # Find dates in px_long but NOT in meta
    # px_dates = set(px_long['date'].unique())
    # meta_dates = set(meta['date'].unique())
    # print("Dates in px_long missing from meta:", px_dates - meta_dates)


C:\Users\SP15548\AppData\Local\Temp\ipykernel_22292\2074245037.py:25: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,sector,gross_return,cumulative_return
8,Materials,1.882192,0.882192
4,Financials,1.677760,0.677760
7,Information Technology,1.675821,0.675821
0,Communication Services,1.513550,0.513550
6,Industrials,1.507415,0.507415
1,Consumer Discretionary,1.438616,0.438616
3,Energy,1.320623,0.320623
10,Utilities,1.274135,0.274135
5,Health Care,1.042666,0.042666
2,Consumer Staples,1.036209,0.036209
